# Fantasy Squads - Wyscout stats meet Fantasy Premier League

An experiment away from Southend: line Wyscout's Premier League players up with the Fantasy
Premier League game, build a table of their season statistics against their FPL cost and
points, and export it for a Ludwig model that predicts what a player *should* cost. The
matching and table-building live in `prem_fl.mapping`.

## Setup

Reload the `wyscout` package on every cell so edits to it are picked up without restarting the kernel, then load the API credentials and point at the local cache. Nothing here produces output.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # CLIENT_ID and CLIENT_SECRET for the Wyscout API, from notebooks/.env
os.environ.setdefault("MONGO_HOST", "localhost")  # every API response is cached in Mongo

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

## IDs

The Premier League season and round the Ludwig model was trained on (2022/23). Change these
to build the table for another season.

In [3]:
from wyscout.ids import PREMIER_LEAGUE, ENGLAND

PL_SEASON = 188095  # Premier League 2022/23
PL_ROUND = 4426845  # the regular-season round of that season

## Find the Premier League in the API

Competitions are listed by area; this is how `PREMIER_LEAGUE` was found.

In [4]:
from wyscout.competitions import get_competitions

[c for c in get_competitions(ENGLAND)["competitions"] if "Premier" in c["name"]]

[{'wyId': 44089,
  'name': 'U16 Premier League Cup',
  'area': {'id': 826,
   'alpha2code': 'EN',
   'alpha3code': 'XEN',
   'name': 'England'},
  'format': 'Domestic cup',
  'type': 'club',
  'category': 'U16',
  'gender': 'male',
  'divisionLevel': 0},
 {'wyId': 44085,
  'name': 'National Football Youth League U21 South Premier 2',
  'area': {'id': 826,
   'alpha2code': 'EN',
   'alpha3code': 'XEN',
   'name': 'England'},
  'format': 'Domestic league',
  'type': 'club',
  'category': 'U21',
  'gender': 'female',
  'divisionLevel': 2},
 {'wyId': 44084,
  'name': 'National Football Youth League U21 South Premier',
  'area': {'id': 826,
   'alpha2code': 'EN',
   'alpha3code': 'XEN',
   'name': 'England'},
  'format': 'Domestic league',
  'type': 'club',
  'category': 'U21',
  'gender': 'female',
  'divisionLevel': 1},
 {'wyId': 44083,
  'name': 'National Football Youth League U21 North Premier',
  'area': {'id': 826,
   'alpha2code': 'EN',
   'alpha3code': 'XEN',
   'name': 'England'},


## The two player lists

FPL's public bootstrap endpoint gives every player with their cost and points; Wyscout's
competition players endpoint gives everyone registered in the league. The Wyscout call is
paginated and takes a few seconds.

In [5]:
from prem_fl import get_details
from prem_fl.mapping import fpl_players, match_players, player_table
from wyscout.competitions import get_all_competition_players

fpl = fpl_players(get_details())
wyscout_players = get_all_competition_players(PREMIER_LEAGUE)
print(f"{len(fpl)} FPL players, {len(wyscout_players)} Wyscout players")

667 FPL players, 491 Wyscout players


## Match them up

A simple exact match on first and last name. It misses players the two sources spell
differently, so the unmatched list is worth a look.

In [6]:
matched, unmatched = match_players(wyscout_players, fpl)
print(f"matched {len(matched)}, unmatched {len(unmatched)}")
[p["shortName"] for p in unmatched[:20]]

matched 353, unmatched 138


['J. Ta Bi',
 'C. Yirenkyi',
 'D. Methalie',
 'M. Zepa',
 'B. Fredrick',
 'M. Yalcouyé',
 'Estevão',
 'Rio Ngumoha',
 'Geovany Quenda',
 'T. Bazoumana',
 'Chema Andrés',
 'B. Burrowes',
 'César Palacios',
 'S. Chérif',
 'J. King',
 'J. Soler',
 'J. Acheampong',
 'Alysson Edward',
 'A. Tóth',
 'C. Echeverri']

## Season statistics against FPL cost and points

One row per matched player: every field from Wyscout's season totals, then FPL position, cost
and points. This fetches each player's stats, so the first run takes a while; after that it
comes from the cache.

In [7]:
df = player_table(matched, fpl, PL_SEASON, PREMIER_LEAGUE, PL_ROUND)
df.head()

,playerId,name,matches,matchesInStart,matchesSubstituted,matchesComingOff,minutesOnField,minutesTagged,goals,assists,...,gkSaves,newDuelsWon,newDefensiveDuelsWon,newOffensiveDuelsWon,newSuccessfulDribbles,lateralPasses,successfulLateralPasses,position,cost,points
0,1213717,Z. Yohanna,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,Midfielders,49,1
1,1191271,J. Ahoka,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,Midfielders,45,0
2,1154197,M. Nicoll-Jazuli,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,Midfielders,45,1
3,1103728,M. Miley,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,Midfielders,45,0
4,1098508,Y. Ibrahim,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,Midfielders,45,0


## Export for Ludwig

The table as CSV for `ludwig/config.yaml`, and the column list in the shape the config wants
(one `- name:` entry per numeric feature). The CSV is rewritten each run, not appended to.

In [8]:
from pathlib import Path

LUDWIG_DIR = Path("../ludwig")
df.to_csv(LUDWIG_DIR / "player_data.csv", index=False)

for col in df.columns:
    print(f"    - name: {col}")
    print("      type: number")

    - name: playerId
      type: number
    - name: name
      type: number
    - name: matches
      type: number
    - name: matchesInStart
      type: number
    - name: matchesSubstituted
      type: number
    - name: matchesComingOff
      type: number
    - name: minutesOnField
      type: number
    - name: minutesTagged
      type: number
    - name: goals
      type: number
    - name: assists
      type: number
    - name: shots
      type: number
    - name: headShots
      type: number
    - name: yellowCards
      type: number
    - name: redCards
      type: number
    - name: directRedCards
      type: number
    - name: penalties
      type: number
    - name: linkupPlays
      type: number
    - name: duels
      type: number
    - name: duelsWon
      type: number
    - name: defensiveDuels
      type: number
    - name: defensiveDuelsWon
      type: number
    - name: offensiveDuels
      type: number
    - name: offensiveDuelsWon
      type: number
    - name: aeri

## Train and predict

Run from the `ludwig/` directory:

```
ludwig train --dataset player_data.csv --config config.yaml
ludwig predict --dataset player_data.csv --model_path=results/experiment_run/model
```

then copy the predicted cost column to `ludwig/results/cost_predictions.csv`. The cell below
only runs when that file exists.

In [9]:
PREDICTIONS = LUDWIG_DIR / "results" / "cost_predictions.csv"

if PREDICTIONS.exists():
    predicted = pd.read_csv(PREDICTIONS, header=None)[0]
    df["predictedCost"] = pd.to_numeric(predicted)
    df["predictedCostDiff"] = df["cost"] - df["predictedCost"]
    print("Most under-priced by the model:")
    display(df.sort_values("predictedCostDiff").head(20)[["name", "position", "cost", "predictedCost", "points"]])
else:
    print(f"No predictions at {PREDICTIONS.resolve()} - train the model first")

No predictions at /Users/chris.hunt/Dev/personal/tools/sufc_stats/ludwig/results/cost_predictions.csv - train the model first


## The top scorers, for reference

In [10]:
df.sort_values("points", ascending=False).head(20)[["name", "position", "cost", "points", "goals", "assists"]]

,name,position,cost,points,goals,assists
164,P. Groß,Midfielders,58,47,9,8
181,J. Tarkowski,Defenders,61,43,1,1
296,J. Bogle,Defenders,46,42,0,0
155,K. Schade,Midfielders,61,39,0,1
139,M. De Cuyper,Defenders,49,38,0,0
128,R. Cherki,Midfielders,78,34,0,0
114,K. Tzolakis,Goalkeepers,46,34,0,0
143,A. Semenyo,Midfielders,84,33,1,0
264,A. Isak,Forwards,91,33,10,1
111,João Pedro,Forwards,78,33,0,0
